In [1]:
from pathlib import Path
import re
import json
import shutil
import warnings
import pandas as pd

# Resolve the project root whether Jupyter starts in the project root or notebooks/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
TEST_FOLDER = DATA_RAW / "test_resumes"
CANDIDATE_FOLDER = DATA_PROCESSED / "candidates"
RESUME_LIBRARY = DATA_PROCESSED / "cv_builder" / "resume_library"

for folder in (TEST_FOLDER, DATA_PROCESSED, CANDIDATE_FOLDER, RESUME_LIBRARY):
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Resume input folder:", TEST_FOLDER)

Project root: /Users/rizwansalim/Desktop/SkillBridgeByZequa
Resume input folder: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/raw/test_resumes


## 1. PDF text extraction

Text-based PDFs are read with PyMuPDF. Scanned PDFs use OCR only when Poppler and Tesseract are installed. On macOS with MacPorts, install OCR tools in Terminal with `sudo port install poppler tesseract`.

In [2]:
try:
    import fitz  # PyMuPDF
except ImportError as exc:
    raise ImportError("Install PyMuPDF in this notebook environment: %pip install pymupdf") from exc

def extract_resume_text(pdf_path):
    """Extract selectable text from a PDF using PyMuPDF."""
    pdf_path = Path(pdf_path).expanduser()
    if not pdf_path.is_file():
        raise FileNotFoundError(f"Resume PDF not found: {pdf_path}")
    pages = []
    with fitz.open(pdf_path) as document:
        for page in document:
            pages.append(page.get_text("text") or "")
    return "\n".join(pages).strip()

def extract_scanned_pdf(pdf_path, dpi=300, lang="eng"):
    """OCR a scanned PDF. Requires pdf2image, Poppler, and Tesseract."""
    from pdf2image import convert_from_path
    import pytesseract

    pdfinfo_exe = shutil.which("pdfinfo")
    pdftoppm_exe = shutil.which("pdftoppm")
    tesseract_exe = shutil.which("tesseract")

    # MacPorts fallback for Jupyter kernels that do not inherit the shell PATH.
    macports_bin = Path("/opt/local/bin")
    pdfinfo_exe = pdfinfo_exe or (str(macports_bin / "pdfinfo") if (macports_bin / "pdfinfo").exists() else None)
    pdftoppm_exe = pdftoppm_exe or (str(macports_bin / "pdftoppm") if (macports_bin / "pdftoppm").exists() else None)
    tesseract_exe = tesseract_exe or (str(macports_bin / "tesseract") if (macports_bin / "tesseract").exists() else None)

    if not pdfinfo_exe or not pdftoppm_exe:
        raise RuntimeError("Poppler is missing. Install it with `sudo port install poppler`, then restart the kernel.")
    if not tesseract_exe:
        raise RuntimeError("Tesseract is missing. Install it with `sudo port install tesseract`, then restart the kernel.")

    pytesseract.pytesseract.tesseract_cmd = tesseract_exe
    images = convert_from_path(
        str(Path(pdf_path).expanduser()),
        poppler_path=str(Path(pdfinfo_exe).parent),
        dpi=dpi
    )
    return "\n".join(
        pytesseract.image_to_string(image, lang=lang)
        for image in images
    ).strip()

def extract_pdf_automatically(pdf_path, min_text_chars=50):
    """Prefer embedded PDF text; fall back to OCR for image-only PDFs."""
    text = extract_resume_text(pdf_path)
    if len(text.strip()) >= min_text_chars:
        return text, "embedded text"
    try:
        ocr_text = extract_scanned_pdf(pdf_path)
    except (ImportError, RuntimeError) as exc:
        if text.strip():
            warnings.warn(f"OCR unavailable; returning the limited embedded text. Details: {exc}")
            return text, "limited embedded text"
        raise RuntimeError(f"No readable PDF text and OCR is unavailable: {exc}") from exc
    if not ocr_text:
        raise ValueError(f"No text could be extracted from {pdf_path}")
    return ocr_text, "OCR"

## 2. Contact details, sections, and skills

The output uses lowercase dictionary keys consistently: `contact.email`, `sections.education`, and so on.

In [3]:
def extract_contact_details(text):
    text = text or ""
    result = {"name": None, "email": None, "phone": None, "linkedin": None, "github": None}

    email = re.search(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", text)
    phone = re.search(r"(?<!\d)(?:\+?\d{1,3}[ .-]?)?(?:\(?\d{2,5}\)?[ .-]?)?\d{5,10}(?!\d)", text)
    linkedin = re.search(r"(?:https?://)?(?:www\.)?linkedin\.com/in/[A-Za-z0-9_-]+", text, re.I)
    github = re.search(r"(?:https?://)?(?:www\.)?github\.com/[A-Za-z0-9_-]+", text, re.I)

    result["email"] = email.group(0) if email else None
    result["phone"] = phone.group(0).strip() if phone else None
    result["linkedin"] = linkedin.group(0) if linkedin else None
    result["github"] = github.group(0) if github else None

    # Conservative name heuristic: first non-empty line, unless it looks like contact info or a heading.
    headings = {"resume", "curriculum vitae", "cv", "profile", "professional summary"}
    first_lines = [line.strip() for line in text.splitlines() if line.strip()]
    if first_lines:
        candidate = first_lines[0]
        if len(candidate.split()) <= 5 and candidate.lower() not in headings and not re.search(r"@|https?://|\d", candidate):
            result["name"] = candidate
    return result


SECTION_ALIASES = {
    "summary": ["professional summary", "summary", "profile", "career objective", "objective"],
    "experience": ["work experience", "professional experience", "employment history", "experience"],
    "education": ["education", "academic background", "qualifications"],
    "skills": ["technical skills", "key skills", "skills", "core competencies"],
    "projects": ["projects", "personal projects", "academic projects"],
    "certifications": ["certifications", "certificates", "licenses"],
    "achievements": ["achievements", "awards", "honors"],
    "languages": ["languages", "language proficiency"]
}
HEADING_TO_SECTION = {
    re.sub(r"[:\s]+$", "", alias.lower()): section
    for section, aliases in SECTION_ALIASES.items()
    for alias in aliases
}

def extract_all_resume_sections(text):
    sections = {name: "" for name in SECTION_ALIASES}
    collected = {name: [] for name in SECTION_ALIASES}
    current = None

    for line in (text or "").splitlines():
        cleaned = line.strip()
        if not cleaned:
            continue
        normalized = re.sub(r"[:\s]+$", "", cleaned.lower())
        # Ignore common markdown/bullet decoration around section headings.
        normalized = re.sub(r"^[#*•\-\s]+", "", normalized)
        normalized = re.sub(r"[:\s]+$", "", normalized)
        matched = HEADING_TO_SECTION.get(normalized)
        if matched:
            current = matched
        elif current:
            collected[current].append(cleaned)

    for section, lines in collected.items():
        sections[section] = "\n".join(lines).strip()
    return sections


DEFAULT_SKILL_NAMES = [
    "Python", "SQL", "Machine Learning", "Deep Learning", "Data Analysis",
    "Pandas", "NumPy", "Data Visualization", "Statistics", "Excel",
    "Power BI", "Tableau", "Java", "JavaScript", "HTML", "CSS",
    "Natural Language Processing", "TensorFlow", "PyTorch", "Git"
]

def load_skill_names():
    """Load the project skill vocabulary; use a small demo vocabulary if absent."""
    candidates = [
        DATA_PROCESSED / "unified_skills.csv",
        DATA_PROCESSED / "unified_skills_standardized.csv"
    ]
    for path in candidates:
        if path.is_file():
            frame = pd.read_csv(path)
            possible = ["Skill Name", "skill_name", "skill", "name"]
            column = next((col for col in possible if col in frame.columns), None)
            if column is None:
                raise ValueError(f"No skill-name column in {path.name}. Columns: {frame.columns.tolist()}")
            names = frame[column].dropna().astype(str).str.strip()
            return sorted(set(name for name in names if name), key=str.lower)
    warnings.warn("Unified skills CSV not found. Using the built-in demo skill vocabulary.")
    return DEFAULT_SKILL_NAMES.copy()

skill_names = load_skill_names()
print(f"Skill vocabulary loaded: {len(skill_names):,} skills")

Skill vocabulary loaded: 13,949 skills


In [4]:
def advanced_skill_extraction(text, skill_names):
    """Match vocabulary terms on word boundaries; return unique source labels."""
    text_lower = (text or "").lower()
    found = []
    for skill in skill_names:
        skill = str(skill).strip()
        if not skill:
            continue
        pattern = r"(?<![a-zA-Z0-9])" + re.escape(skill.lower()) + r"(?![a-zA-Z0-9])"
        if re.search(pattern, text_lower):
            found.append(skill)
    return sorted(set(found), key=str.lower)


def process_resume(pdf_path):
    pdf_path = Path(pdf_path).expanduser()
    raw_text, extraction_method = extract_pdf_automatically(pdf_path)
    if not raw_text.strip():
        raise ValueError(f"No resume text extracted from {pdf_path}")

    contact = extract_contact_details(raw_text)
    sections = extract_all_resume_sections(raw_text)
    skills = advanced_skill_extraction(raw_text, skill_names)

    return {
        "file_name": pdf_path.name,
        "file_path": str(pdf_path.resolve()),
        "extraction_method": extraction_method,
        "contact": contact,
        "sections": sections,
        "skills": skills,
        "raw_text": raw_text
    }

print("Resume parser functions are ready.")

Resume parser functions are ready.


## 3. Create or select a test PDF

The next cell creates a clearly labelled synthetic test resume only if the input folder contains no PDFs. You can replace it with your own PDF at any time.

In [5]:
pdf_files = sorted(TEST_FOLDER.glob("*.pdf"))

if not pdf_files:
    # PyMuPDF is already required above, so no extra PDF-generation dependency is needed.
    sample_pdf = TEST_FOLDER / "sample_resume.pdf"
    document = fitz.open()
    page = document.new_page()
    lines = [
        "Rizwan Salim",
        "Email: rizwan@example.com",
        "Phone: +91 9876543210",
        "LinkedIn: linkedin.com/in/rizwansalim",
        "",
        "PROFESSIONAL SUMMARY",
        "Entry-level data professional interested in Python, SQL, and machine learning.",
        "",
        "SKILLS",
        "Python, SQL, Machine Learning, Data Analysis, Pandas, NumPy",
        "",
        "EDUCATION",
        "Bachelor of Computer Applications",
        "",
        "PROJECTS",
        "Built a data analysis project using Python and Pandas.",
        "",
        "EXPERIENCE",
        "Academic and personal project experience."
    ]
    y = 780
    for line in lines:
        page.insert_text((55, y), line[:110], fontsize=11)
        y -= 24
    document.save(sample_pdf)
    document.close()
    pdf_files = [sample_pdf]
    print("Created synthetic test PDF:", sample_pdf)
else:
    print("Found PDF files:")
    for path in pdf_files:
        print(" -", path.name)

Found PDF files:
 - sample_resume.pdf
 - scanned_test_resume.pdf


## 4. Process one resume and validate the output

In [6]:
if not pdf_files:
    raise FileNotFoundError(f"Add at least one PDF resume to {TEST_FOLDER} before continuing.")

selected_pdf = pdf_files[0]
updated_result = process_resume(selected_pdf)

print("Processed:", updated_result["file_name"])
print("Extraction method:", updated_result["extraction_method"])
print("Candidate name:", updated_result["contact"]["name"])
print("Email:", updated_result["contact"]["email"])
print("Phone:", updated_result["contact"]["phone"])
print("Skills found:", len(updated_result["skills"]))
print("Section availability:", {key: bool(value) for key, value in updated_result["sections"].items()})

Processed: sample_resume.pdf
Extraction method: OCR
Candidate name: RIZWAN SALIM
Email: rizwan@example.com
Phone: 9876543210
Skills found: 5
Section availability: {'summary': False, 'experience': True, 'education': True, 'skills': True, 'projects': True, 'certifications': True, 'achievements': False, 'languages': False}


In [7]:
def validate_resume(resume_data):
    contact = resume_data.get("contact", {})
    sections = resume_data.get("sections", {})
    skills = resume_data.get("skills", [])
    checks = {
        "Email Available": bool(contact.get("email")),
        "Phone Available": bool(contact.get("phone")),
        "Skills Available": bool(skills),
        "Education Available": bool(sections.get("education")),
        "Experience Available": bool(sections.get("experience")),
        "Projects Available": bool(sections.get("projects"))
    }
    total = len(checks)
    passed = sum(checks.values())
    return {"checks": checks, "passed": passed, "total": total, "score": round(passed / total * 100, 2)}

validation = validate_resume(updated_result)
print("Resume validation:", validation["score"], "%")
display(pd.DataFrame(validation["checks"].items(), columns=["Check", "Passed"]))

Resume validation: 100.0 %


,Check,Passed
0,Email Available,True
1,Phone Available,True
2,Skills Available,True
3,Education Available,True
4,Experience Available,True
5,Projects Available,True


## 5. Standard candidate profile and resume library

This schema is intentionally consistent and JSON-serializable. The original resume is stored separately from any later role-tailored CV versions.

In [8]:
def create_candidate_profile(resume_data):
    contact = resume_data.get("contact", {})
    sections = resume_data.get("sections", {})
    return {
        "candidate_id": Path(resume_data["file_name"]).stem,
        "name": contact.get("name"),
        "email": contact.get("email"),
        "phone": contact.get("phone"),
        "linkedin": contact.get("linkedin"),
        "github": contact.get("github"),
        "skills": list(resume_data.get("skills", [])),
        "summary": sections.get("summary", ""),
        "education": sections.get("education", ""),
        "experience": sections.get("experience", ""),
        "projects": sections.get("projects", ""),
        "certifications": sections.get("certifications", ""),
        "achievements": sections.get("achievements", ""),
        "languages": sections.get("languages", ""),
        "source_file": resume_data.get("file_path"),
        "extraction_method": resume_data.get("extraction_method")
    }

candidate_profile = create_candidate_profile(updated_result)
print(json.dumps(candidate_profile, indent=2, ensure_ascii=False))

{
  "candidate_id": "sample_resume",
  "name": "RIZWAN SALIM",
  "email": "rizwan@example.com",
  "phone": "9876543210",
  "linkedin": null,
  "github": null,
  "skills": [
    "computer science",
    "deep learning",
    "machine learning",
    "Science",
    "SQL"
  ],
  "summary": "",
  "education": "B.Tech Computer Science\nABC Engineering College",
  "experience": "Python Developer Intern\nDeveloped machine learning applications.\nWorked with Python, SQL and Pandas.",
  "projects": "Customer Churn Prediction\nBuilt a machine learning classification model.",
  "certifications": "Python Programming Certification",
  "achievements": "",
  "languages": "",
  "source_file": "/Users/rizwansalim/Desktop/SkillBridgeByZequa/data/raw/test_resumes/sample_resume.pdf",
  "extraction_method": "OCR"
}


In [9]:
def save_resume_json(profile, output_dir=RESUME_LIBRARY):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    candidate_id = profile.get("candidate_id") or "candidate"
    output_path = output_dir / f"{candidate_id}.json"
    with output_path.open("w", encoding="utf-8") as file:
        json.dump(profile, file, indent=2, ensure_ascii=False)
    return output_path

saved_profile_path = save_resume_json(candidate_profile)
print("Original candidate profile saved:", saved_profile_path)

Original candidate profile saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/cv_builder/resume_library/sample_resume.json


## 6. Batch processing and candidate registry

In [10]:
batch_results = []
batch_errors = []

for pdf_path in sorted(TEST_FOLDER.glob("*.pdf")):
    try:
        batch_results.append(process_resume(pdf_path))
    except Exception as exc:
        batch_errors.append({"file": pdf_path.name, "error": str(exc)})

print("Successfully processed:", len(batch_results))
print("Failed:", len(batch_errors))
if batch_errors:
    display(pd.DataFrame(batch_errors))

Successfully processed: 2
Failed: 0


In [11]:
candidate_records = []
for result in batch_results:
    contact = result.get("contact", {})
    sections = result.get("sections", {})
    candidate_records.append({
        "File Name": result["file_name"],
        "Name": contact.get("name") or "",
        "Email": contact.get("email") or "",
        "Phone": contact.get("phone") or "",
        "Total Skills": len(result.get("skills", [])),
        "Skills": ", ".join(result.get("skills", [])),
        "Education Found": bool(sections.get("education")),
        "Experience Found": bool(sections.get("experience")),
        "Projects Found": bool(sections.get("projects"))
    })

batch_summary_df = pd.DataFrame(candidate_records)
display(batch_summary_df)
batch_summary_path = CANDIDATE_FOLDER / "batch_resume_results.csv"
batch_summary_df.to_csv(batch_summary_path, index=False)
print("Batch summary saved:", batch_summary_path)

,File Name,Name,Email,Phone,Total Skills,Skills,Education Found,Experience Found,Projects Found
0,sample_resume.pdf,RIZWAN SALIM,rizwan@example.com,9876543210,5,"computer science, deep learning, machine learn...",True,True,True
1,scanned_test_resume.pdf,RIZWAN SALIM,rizwan@example.com,9876543210,5,"computer science, deep learning, machine learn...",True,True,True


Batch summary saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/candidates/batch_resume_results.csv


## 7. ESCO-based career recommendations (optional dataset)

Expected inputs:
- `data/processed/esco_occupation_skill_relationships.csv` with occupation and skill labels/URIs, or
- equivalent data containing `occupationLabel` and `skillLabel`.

If the file is missing or its schema differs, the notebook reports that career matching is skipped; resume parsing remains usable.

In [12]:
ESCO_RELATIONS_PATH = DATA_PROCESSED / "esco_occupation_skill_relationships.csv"
occupation_skill_mapping = pd.DataFrame()

if ESCO_RELATIONS_PATH.is_file():
    esco_relations_df = pd.read_csv(ESCO_RELATIONS_PATH)
    column_aliases = {
        "occupationUri": "occupation_id",
        "occupationLabel": "occupation_name",
        "skillUri": "skill_id",
        "skillLabel": "skill_name",
        "skillType": "skill_type"
    }
    available = {source: target for source, target in column_aliases.items() if source in esco_relations_df.columns}
    occupation_skill_mapping = esco_relations_df.rename(columns=available).copy()
    required = {"occupation_name", "skill_name"}
    if not required.issubset(occupation_skill_mapping.columns):
        warnings.warn(f"ESCO file missing required columns {sorted(required)}. Career matching will be skipped.")
        occupation_skill_mapping = pd.DataFrame()
    else:
        if "skill_type" not in occupation_skill_mapping:
            occupation_skill_mapping["skill_type"] = ""
        occupation_skill_mapping["weight"] = occupation_skill_mapping["skill_type"].fillna("").astype(str).str.lower().map(
            lambda value: 2.0 if "essential" in value else 1.0
        )
        print("ESCO relationship rows:", len(occupation_skill_mapping))
else:
    print("ESCO relationship CSV not found; career recommendations will be skipped.")

ESCO relationship rows: 126051


In [13]:
def recommend_careers(candidate_skills, mapping, top_n=5):
    if mapping.empty:
        return pd.DataFrame(columns=["Career", "Match Score", "Matched Skills", "Missing Skills"])
    candidate_set = {str(skill).strip().lower() for skill in candidate_skills if str(skill).strip()}
    rows = []
    for occupation, group in mapping.groupby("occupation_name"):
        required = sorted(set(group["skill_name"].dropna().astype(str).str.strip().str.lower()) - {""})
        if not required:
            continue
        matched = sorted(candidate_set.intersection(required))
        missing = sorted(set(required) - candidate_set)
        rows.append({
            "Career": occupation,
            "Match Score": round(len(matched) / len(required) * 100, 2),
            "Matched Skills": matched,
            "Missing Skills": missing,
            "Total Required Skills": len(required)
        })
    return pd.DataFrame(rows).sort_values("Match Score", ascending=False).head(top_n).reset_index(drop=True) if rows else pd.DataFrame()

career_recommendations = recommend_careers(updated_result["skills"], occupation_skill_mapping)
if career_recommendations.empty:
    print("No career recommendations generated. Check the ESCO dataset and whether extracted skills match its skill labels.")
else:
    display(career_recommendations)
    career_recommendations.to_csv(CANDIDATE_FOLDER / "career_recommendations.csv", index=False)

,Career,Match Score,Matched Skills,Missing Skills,Total Required Skills
0,IoT developer,9.09,"[computer science, machine learning]","[algorithms, analyse big data, computer techno...",22
1,battery simulation engineer,7.69,[computer science],"[algorithms, battery design, computer programm...",13
2,computer vision engineer,3.85,"[deep learning, machine learning]","[apply statistical analysis techniques, cognit...",52
3,battery system engineer,3.70,[computer science],"[analyse test data, battery chemistry, battery...",27
4,data engineer,3.57,[computer science],"[analyse pipeline database information, cloud ...",28


## 8. Skill-gap analysis and learning roadmap

In [14]:
def analyze_career_skill_gap(candidate_skills, mapping, career_name):
    candidate_set = {str(skill).strip().lower() for skill in candidate_skills}
    career_rows = mapping[mapping["occupation_name"] == career_name]
    required = sorted(set(career_rows["skill_name"].dropna().astype(str).str.strip().str.lower()) - {""})
    matched = sorted(candidate_set.intersection(required))
    missing = sorted(set(required) - candidate_set)
    coverage = len(matched) / len(required) * 100 if required else 0
    return {
        "Target Career": career_name,
        "Existing Skills": matched,
        "Missing Skills": missing,
        "Total Required Skills": len(required),
        "Matched Skills Count": len(matched),
        "Missing Skills Count": len(missing),
        "Skill Coverage Percentage": round(coverage, 2)
    }

learning_resources = {
    "python": {"resource": "Python for Everybody", "url": "https://www.py4e.com/", "platform": "University of Michigan"},
    "sql": {"resource": "SQL Tutorial", "url": "https://www.w3schools.com/sql/", "platform": "W3Schools"},
    "machine learning": {"resource": "Machine Learning Specialization", "url": "https://www.coursera.org/specializations/machine-learning-introduction", "platform": "Coursera"},
    "deep learning": {"resource": "Deep Learning Specialization", "url": "https://www.coursera.org/specializations/deep-learning", "platform": "Coursera"},
    "statistics": {"resource": "Statistics Course", "url": "https://www.khanacademy.org/math/statistics-probability", "platform": "Khan Academy"},
    "data visualization": {"resource": "Data Visualization Training", "url": "https://www.tableau.com/learn/training", "platform": "Tableau"}
}
skill_priority = {
    "python": ("High", 20), "sql": ("High", 15), "statistics": ("High", 15),
    "machine learning": ("High", 25), "deep learning": ("Medium", 30),
    "data visualization": ("Medium", 12)
}

def generate_learning_roadmap(skill_gap):
    rows = []
    for skill in skill_gap.get("Missing Skills", []):
        resource = learning_resources.get(skill.lower(), {
            "resource": f"Search learning materials for {skill}",
            "url": f"https://www.google.com/search?q={skill.replace(' ', '+')}+free+course",
            "platform": "Online resources"
        })
        priority, hours = skill_priority.get(skill.lower(), ("Medium", 10))
        rows.append({
            "Skill": skill,
            "Recommended Resource": resource["resource"],
            "Platform": resource["platform"],
            "URL": resource["url"],
            "Priority": priority,
            "Estimated Hours": hours
        })
    if not rows:
        return pd.DataFrame(columns=["Skill", "Recommended Resource", "Platform", "URL", "Priority", "Estimated Hours"])
    priority_order = {"High": 0, "Medium": 1, "Low": 2}
    frame = pd.DataFrame(rows)
    frame["_priority_order"] = frame["Priority"].map(priority_order)
    return frame.sort_values(["_priority_order", "Estimated Hours"]).drop(columns="_priority_order").reset_index(drop=True)

if not career_recommendations.empty:
    target_career = career_recommendations.iloc[0]["Career"]
    skill_gap_result = analyze_career_skill_gap(updated_result["skills"], occupation_skill_mapping, target_career)
    prioritized_roadmap = generate_learning_roadmap(skill_gap_result)
    print("Target career:", target_career)
    print("Skill coverage:", skill_gap_result["Skill Coverage Percentage"], "%")
    display(prioritized_roadmap)
    with (CANDIDATE_FOLDER / "personalized_skill_gap.json").open("w", encoding="utf-8") as file:
        json.dump(skill_gap_result, file, indent=2)
    prioritized_roadmap.to_csv(CANDIDATE_FOLDER / "prioritized_learning_roadmap.csv", index=False)
else:
    skill_gap_result = None
    prioritized_roadmap = pd.DataFrame()
    print("Skill-gap analysis skipped because there are no career recommendations.")

Target career: IoT developer
Skill coverage: 9.09 %


,Skill,Recommended Resource,Platform,URL,Priority,Estimated Hours
0,algorithms,Search learning materials for algorithms,Online resources,https://www.google.com/search?q=algorithms+fre...,Medium,10
1,analyse big data,Search learning materials for analyse big data,Online resources,https://www.google.com/search?q=analyse+big+da...,Medium,10
2,computer technology,Search learning materials for computer technology,Online resources,https://www.google.com/search?q=computer+techn...,Medium,10
3,design information system,Search learning materials for design informati...,Online resources,https://www.google.com/search?q=design+informa...,Medium,10
4,develop ict workflow,Search learning materials for develop ict work...,Online resources,https://www.google.com/search?q=develop+ict+wo...,Medium,10
5,digital data processing,Search learning materials for digital data pro...,Online resources,https://www.google.com/search?q=digital+data+p...,Medium,10
6,establish data processes,Search learning materials for establish data p...,Online resources,https://www.google.com/search?q=establish+data...,Medium,10
7,hardware testing methods,Search learning materials for hardware testing...,Online resources,https://www.google.com/search?q=hardware+testi...,Medium,10
8,ict architectural frameworks,Search learning materials for ict architectura...,Online resources,https://www.google.com/search?q=ict+architectu...,Medium,10
9,ict software specifications,Search learning materials for ict software spe...,Online resources,https://www.google.com/search?q=ict+software+s...,Medium,10


## 9. Optional semantic skill matching

This section is optional because it downloads a model on first use and requires `sentence-transformers` plus PyTorch. TensorFlow is disabled before importing Transformers to avoid the Python 3.12 `distutils` import problem seen in the earlier notebook. Exact skill matching and the rest of this notebook do not depend on semantic matching.

In [15]:
ENABLE_SEMANTIC_MATCHING = False  # Change to True when sentence-transformers and PyTorch are installed.

semantic_matches_df = pd.DataFrame(columns=["Candidate Skill", "Related ESCO Skill", "Similarity"])

if ENABLE_SEMANTIC_MATCHING and not occupation_skill_mapping.empty and updated_result["skills"]:
    import os
    os.environ["USE_TF"] = "0"
    os.environ["USE_FLAX"] = "0"
    try:
        from sentence_transformers import SentenceTransformer, util
        semantic_model = SentenceTransformer("all-MiniLM-L6-v2")
        candidate_skills = updated_result["skills"]
        occupation_skills = occupation_skill_mapping["skill_name"].dropna().astype(str).unique().tolist()
        candidate_embeddings = semantic_model.encode(candidate_skills, convert_to_tensor=True, normalize_embeddings=True)
        occupation_embeddings = semantic_model.encode(occupation_skills, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True)
        similarities = util.cos_sim(candidate_embeddings, occupation_embeddings)
        matches = []
        for row_index, candidate_skill in enumerate(candidate_skills):
            k = min(3, len(occupation_skills))
            for index in similarities[row_index].topk(k=k).indices.tolist():
                matches.append({
                    "Candidate Skill": candidate_skill,
                    "Related ESCO Skill": occupation_skills[index],
                    "Similarity": round(float(similarities[row_index, index]), 4)
                })
        semantic_matches_df = pd.DataFrame(matches)
        semantic_matches_df.to_csv(CANDIDATE_FOLDER / "semantic_skill_matches.csv", index=False)
        display(semantic_matches_df)
    except Exception as exc:
        warnings.warn(f"Semantic matching was skipped: {exc}")
else:
    print("Semantic matching disabled or its required inputs are unavailable.")

Semantic matching disabled or its required inputs are unavailable.


## 10. Export a unified candidate career report

In [16]:
candidate_career_report = {
    "candidate_profile": candidate_profile,
    "resume_validation": validation,
    "career_recommendations": career_recommendations.to_dict(orient="records"),
    "skill_gap_analysis": skill_gap_result,
    "learning_roadmap": prioritized_roadmap.to_dict(orient="records"),
    "semantic_skill_matches": semantic_matches_df.to_dict(orient="records")
}

report_path = CANDIDATE_FOLDER / "complete_candidate_career_report.json"
with report_path.open("w", encoding="utf-8") as file:
    json.dump(candidate_career_report, file, indent=2, ensure_ascii=False)

print("Complete candidate career report saved:", report_path)

Complete candidate career report saved: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/candidates/complete_candidate_career_report.json


In [17]:
# Final integrity checks: these validate the expected schema and generated artifacts.
assert isinstance(updated_result, dict)
assert set(["file_name", "contact", "sections", "skills", "raw_text"]).issubset(updated_result)
assert set(["name", "email", "phone", "linkedin", "github"]).issubset(updated_result["contact"])
assert all(key in updated_result["sections"] for key in SECTION_ALIASES)
assert isinstance(updated_result["skills"], list)
assert saved_profile_path.is_file()
assert report_path.is_file()

print("Notebook workflow checks passed.")
print("Resume JSON:", saved_profile_path)
print("Career report:", report_path)
print("Resume parser is ready for downstream notebooks.")

Notebook workflow checks passed.
Resume JSON: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/cv_builder/resume_library/sample_resume.json
Career report: /Users/rizwansalim/Desktop/SkillBridgeByZequa/data/processed/candidates/complete_candidate_career_report.json
Resume parser is ready for downstream notebooks.
